# CODI Training on GPU - Simplified Version

**No CODI repo cloning needed!** Everything is bundled.

**GPU**: Free T4 → Runtime → Change runtime type → T4 GPU

**Time**: ~2 hours for 3 epochs on 6,000 examples

In [ ]:
# Verify GPU is active
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Clone your repo (includes codi_bundle with everything needed)
!git clone https://github.com/nabilanewaz/TokenSkip.git
%cd TokenSkip
!ls -lh codi_bundle/

In [ ]:
# Install dependencies
!pip install peft==0.15.2 datasets==3.6.0 transformers==4.52.4 accelerate==1.7.0 safetensors -q
print("✅ Dependencies installed")

In [ ]:
# Verify training data
import json
train_file = 'datasets/gsm8k_split/llm_train.jsonl'
with open(train_file) as f:
    train_count = sum(1 for _ in f)
print(f"✅ Training data: {train_count} examples")

# Show first example
with open(train_file) as f:
    example = json.loads(f.readline())
print(f"\nExample: {example['question'][:100]}...")
print(f"Has CoT: {'cot' in example}")

In [ ]:
# Download CODI checkpoint (GPT-2 + latent projection)
from huggingface_hub import snapshot_download
import os

ckpt_dir = snapshot_download(
    repo_id="zen-E/CODI-gpt2",
    ignore_patterns=["*.msgpack", "*.h5"]
)
print(f"✅ Checkpoint: {ckpt_dir}")

In [ ]:
# Prepare training data in CODI format
%cd codi_bundle

# Create datasets folder
!mkdir -p datasets/gsm8k
!cp ../datasets/gsm8k_split/llm_train.jsonl datasets/gsm8k/train.jsonl
!cp ../datasets/gsm8k_split/validation.jsonl datasets/gsm8k/val.jsonl
print("✅ Data prepared")

In [ ]:
# Start training (adjust batch_size based on GPU memory)
# T4: batch_size=4, A100: batch_size=8+
!python train.py \
  --model_name_or_path gpt2 \
  --seed 42 \
  --model_max_length 512 \
  --lora_r 128 \
  --lora_alpha 32 \
  --lora_init \
  --num_latent 6 \
  --use_prj True \
  --prj_dim 768 \
  --inf_latent_iterations 6 \
  --remove_eos True \
  --use_lora True \
  --batch_size 4 \
  --data_name custom_local:datasets/gsm8k/train.jsonl \
  --output_dir ../outputs/codi_trained \
  --num_train_epochs 3 \
  --learning_rate 0.0002

In [ ]:
# Monitor progress (run this cell repeatedly)
!tail -n 30 ../outputs/codi_trained/*/logs.txt 2>/dev/null || echo "Check for log files in outputs/"

In [ ]:
# After training: check what was saved
%cd ..
!ls -lh outputs/codi_trained/

In [ ]:
# Download checkpoint
!zip -r codi_checkpoint.zip outputs/codi_trained/
from google.colab import files
files.download('codi_checkpoint.zip')
print("✅ Checkpoint download started!")

## Next: Phase 2 - Truth Vector Extraction

After downloading the checkpoint:

```powershell
# Extract locally
Expand-Archive codi_checkpoint.zip

# Phase 2: Extract truth vector from 500 steer examples
python extract_truth_vector.py --steer-data datasets/gsm8k_split/steer_train.jsonl
```